# 04 — NetworkIDBD mathematical derivation: source gate

## Goal

Determine whether the exact NetworkIDBD mathematics can be recovered from an authoritative
source before any optimizer code is written. This is a scientific stop gate, not an
implementation notebook.

**Status: `BLOCKED_UNRESOLVED_FROM_SOURCE`.** The public Sutton/Javed Oak Lab note names
NetworkIDBD and shows its qualitative NoisyMNIST result, but it does not publish the
optimizer equations or enough state definitions to reproduce them.


## Source audit

Sources audited on **2026-08-25**:

1. [Sutton & Javed, *Learning from experience instead of curated datasets*](https://www.oaklab.ai/posts/learning-from-experience-instead-of-curated-datasets)
   is the target public note. It defines NoisyMNIST, the network widths, ReLU, weight
   initialization, the two optimizer names, and the visualization threshold. It contains no
   NetworkIDBD equations, pseudocode, optimizer-state definitions, or hyperparameter table.
2. The [official Oak Lab GitHub organization](https://github.com/oaklab-ai) reported no public
   repositories at audit time, and the note links no supplement or implementation.
3. Sutton's 1992 IDBD paper is an authoritative source for the historical **linear** algorithm,
   but it cannot establish the unpublished neural-network extension used in this experiment.
4. [Degris et al., *Step-size Optimization for Continual Learning*](https://arxiv.org/abs/2401.17401)
   is relevant background on IDBD and step-size vectors. It is not identified by the target note
   as the definition of NetworkIDBD and therefore is not substituted for the missing method.

No generic IDBD, stochastic meta-descent, or remembered update rule is treated as
NetworkIDBD. Similarity of names is not evidence of mathematical identity.


## Paper-defined decisions

The target note explicitly confirms only the following Phase 4-relevant facts:

- NetworkIDBD is described as an extension of IDBD to neural networks.
- It is used to learn the NoisyMNIST network's weight parameters.
- The network has 4,096 inputs, 10,000 ReLU hidden units, and one prediction unit.
- Input-to-hidden weights start uniformly in `(-0.0001, 0.0001)` and hidden-to-output
  weights start at zero.

Those statements do not determine an optimizer update.

## Reproduction-protocol decisions

- Require a source-complete symbolic derivation before allocating adaptive state.
- Treat every absent equation or hyperparameter as `UNRESOLVED_FROM_SOURCE`.
- Compute only conditional memory costs per hypothetical full-size float32 state tensor.
- Keep the official MNIST test partition embargoed; this notebook loads no dataset.

## Open questions

The exact loss, update equations, step-size granularity, adaptive state, initialization,
nonlinear derivative treatment, safeguards, and experiment hyperparameters all remain
`UNRESOLVED_FROM_SOURCE`. The complete itemized inventory appears below.

## ABLATION_OR_EXTENSION

A future experiment using linear IDBD, SMD, or our own neural meta-gradient rule would be an
explicitly named extension. It must not be presented as the paper's NetworkIDBD.


In [1]:
from __future__ import annotations

import platform
from pathlib import Path

import pandas as pd
import psutil
import torch

print(f"Python:  {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
print(f"RAM available at audit: {psutil.virtual_memory().available / 2**30:.2f} GiB")


Python:  3.12.14
PyTorch: 2.13.0+cpu
CUDA:    False
RAM available at audit: 2.27 GiB


## Local implementation and document guard

The checks below make the stop condition auditable: the workspace contains no local PDF/TeX
supplement and `src/` contains no NetworkIDBD implementation. Notebook 01 remains untouched,
and this phase does not import or construct any MNIST partition.


In [2]:
local_math_sources = sorted(
    str(path)
    for path in Path(".").rglob("*")
    if path.is_file() and path.suffix.lower() in {".pdf", ".tex"}
)
implementation_hits = []
for source_path in Path("src").rglob("*.py"):
    if "networkidbd" in source_path.read_text(encoding="utf-8").lower():
        implementation_hits.append(str(source_path))

no_local_mathematical_source_pass = len(local_math_sources) == 0
no_networkidbd_implementation_pass = len(implementation_hits) == 0
assert no_local_mathematical_source_pass
assert no_networkidbd_implementation_pass
print("Local PDF/TeX mathematical sources:", local_math_sources)
print("NetworkIDBD implementation hits in src/:", implementation_hits)


Local PDF/TeX mathematical sources: []
NetworkIDBD implementation hits in src/: []


## Exact-equation audit

The table enumerates the information required by the reproduction plan. No symbolic update is
written for an unresolved item: doing so would silently turn a guess into a paper claim.


In [3]:
equation_audit = pd.DataFrame(
    [
        ("prediction function", "PARTIALLY_PAPER_DEFINED", "architecture known; bias handling unresolved"),
        ("NoisyMNIST loss/error definition", "UNRESOLVED_FROM_SOURCE", "loss scaling and exact equation absent"),
        ("ordinary weight update", "UNRESOLVED_FROM_SOURCE", "NetworkIDBD update absent"),
        ("adaptive step-size granularity", "UNRESOLVED_FROM_SOURCE", "per-weight/feature/unit/layer unknown"),
        ("step-size representation", "UNRESOLVED_FROM_SOURCE", "direct alpha vs another representation unknown"),
        ("log-step-size beta", "UNRESOLVED_FROM_SOURCE", "parameterization and update absent"),
        ("alpha-beta relationship", "UNRESOLVED_FROM_SOURCE", "no exp(beta) relationship stated"),
        ("meta-step-size", "UNRESOLVED_FROM_SOURCE", "symbol, value, or schedule absent"),
        ("eligibility/meta traces", "UNRESOLVED_FROM_SOURCE", "variables and dimensions absent"),
        ("trace update", "UNRESOLVED_FROM_SOURCE", "recurrence and initialization absent"),
        ("normalization terms", "UNRESOLVED_FROM_SOURCE", "none specified"),
        ("safeguards and clipping", "UNRESOLVED_FROM_SOURCE", "none specified"),
        ("adaptive-state initialization", "UNRESOLVED_FROM_SOURCE", "beta/alpha/trace initial values absent"),
        ("nonlinear/ReLU derivative treatment", "UNRESOLVED_FROM_SOURCE", "exact/approximate meta-gradient unknown"),
        ("layer-specific behavior", "UNRESOLVED_FROM_SOURCE", "which layers adapt and how is unknown"),
        ("approximations", "UNRESOLVED_FROM_SOURCE", "no truncation or independence assumption stated"),
        ("distinction from linear IDBD", "UNRESOLVED_FROM_SOURCE", "the extension is named but not defined"),
    ],
    columns=["required_item", "status", "evidence/consequence"],
)
display(equation_audit)
unresolved_count = int(
    equation_audit["status"].eq("UNRESOLVED_FROM_SOURCE").sum()
)
assert unresolved_count == 16
print(f"Unresolved required mathematical items: {unresolved_count}/17")


,required_item,status,evidence/consequence
0,prediction function,PARTIALLY_PAPER_DEFINED,architecture known; bias handling unresolved
1,NoisyMNIST loss/error definition,UNRESOLVED_FROM_SOURCE,loss scaling and exact equation absent
2,ordinary weight update,UNRESOLVED_FROM_SOURCE,NetworkIDBD update absent
3,adaptive step-size granularity,UNRESOLVED_FROM_SOURCE,per-weight/feature/unit/layer unknown
4,step-size representation,UNRESOLVED_FROM_SOURCE,direct alpha vs another representation unknown
5,log-step-size beta,UNRESOLVED_FROM_SOURCE,parameterization and update absent
6,alpha-beta relationship,UNRESOLVED_FROM_SOURCE,no exp(beta) relationship stated
7,meta-step-size,UNRESOLVED_FROM_SOURCE,"symbol, value, or schedule absent"
8,eligibility/meta traces,UNRESOLVED_FROM_SOURCE,variables and dimensions absent
9,trace update,UNRESOLVED_FROM_SOURCE,recurrence and initialization absent


Unresolved required mathematical items: 16/17


## Critical adaptive-step-size question

`UNRESOLVED_FROM_SOURCE`: the public note does not say whether the learned step size belongs
to each of the 40,970,000 weights, each input feature, each hidden unit, each layer, or another
group. This choice changes the algorithm, tensor shapes, memory cost, and scientific claim.
It is therefore a hard implementation gate.

## Conditional computational feasibility

Without knowing the adaptive-state structure, exact memory cannot be reported. The only safe
calculation is conditional: the no-bias model has 40,970,000 float32 parameters, so every
additional parameter-sized state tensor costs 163,880,000 bytes (156.29 MiB). The table below
does not assert how many tensors NetworkIDBD uses.


In [4]:
parameter_count = 40_970_000
float32_bytes = torch.tensor([], dtype=torch.float32).element_size()
one_full_state_bytes = parameter_count * float32_bytes
parameter_and_gradient_bytes = 2 * one_full_state_bytes

conditional_memory = pd.DataFrame(
    [
        {
            "hypothetical_extra_parameter_sized_states": state_count,
            "parameters_plus_gradients_MiB": parameter_and_gradient_bytes / 2**20,
            "extra_state_MiB": state_count * one_full_state_bytes / 2**20,
            "subtotal_MiB": (parameter_and_gradient_bytes + state_count * one_full_state_bytes) / 2**20,
        }
        for state_count in range(7)
    ]
)
display(conditional_memory)
assert one_full_state_bytes == 163_880_000
print(f"Each hypothetical full-size float32 state: {one_full_state_bytes / 2**20:.2f} MiB")
print("Exact NetworkIDBD state count: UNRESOLVED_FROM_SOURCE")


,hypothetical_extra_parameter_sized_states,parameters_plus_gradients_MiB,extra_state_MiB,subtotal_MiB
0,0,312.576294,0.000000,312.576294
1,1,312.576294,156.288147,468.864441
2,2,312.576294,312.576294,625.152588
3,3,312.576294,468.864441,781.440735
4,4,312.576294,625.152588,937.728882
5,5,312.576294,781.440735,1094.017029
6,6,312.576294,937.728882,1250.305176


Each hypothetical full-size float32 state: 156.29 MiB
Exact NetworkIDBD state count: UNRESOLVED_FROM_SOURCE


## Derivative sanity check

A toy-network autograd or finite-difference check cannot be designed until the target update,
meta-objective, traces, and approximation rules are known. Running a check against a generic
IDBD formula would validate a different algorithm. This step is intentionally deferred, not
passed.

## Verification checklist


In [5]:
phase4_checks = [
    ("target Oak Lab note audited", True),
    ("official public-code location audited", True),
    ("required equation inventory completed", len(equation_audit) == 17),
    ("adaptive step-size granularity identified", False),
    ("exact NetworkIDBD update recovered", False),
    ("exact adaptive state and initialization recovered", False),
    ("exact memory requirement computable", False),
    ("toy derivative check safe to perform", False),
    ("no NetworkIDBD implementation was guessed", no_networkidbd_implementation_pass),
    ("official test partition remained untouched", True),
]
for check_name, resolved in phase4_checks:
    label = "PASS" if resolved else "BLOCKED"
    print(f"- [{label}] {check_name}")

safe_to_implement_networkidbd = False
phase4_status = "BLOCKED_UNRESOLVED_FROM_SOURCE"
assert not safe_to_implement_networkidbd
assert phase4_status == "BLOCKED_UNRESOLVED_FROM_SOURCE"
print(f"\nPhase 4 status: {phase4_status}")


- [PASS] target Oak Lab note audited
- [PASS] official public-code location audited
- [PASS] required equation inventory completed
- [BLOCKED] adaptive step-size granularity identified
- [BLOCKED] exact NetworkIDBD update recovered
- [BLOCKED] exact adaptive state and initialization recovered
- [BLOCKED] exact memory requirement computable
- [BLOCKED] toy derivative check safe to perform
- [PASS] no NetworkIDBD implementation was guessed
- [PASS] official test partition remained untouched

Phase 4 status: BLOCKED_UNRESOLVED_FROM_SOURCE


## Results

The source gate is functioning as intended: Phase 4 cannot produce a source-verified
derivation from the material currently available. No optimizer, adaptive tensor, test stream,
or toy surrogate was created.

## Deviations from paper

There is no implemented deviation because implementation stopped. Conditional memory rows are
capacity illustrations, not claims about NetworkIDBD. The Phase 3 smoke loss and learning rate
also remain unresolved reproduction assumptions rather than paper-defined values.

## Required source and next phase

To resume faithfully, obtain one of the following from Sutton, Javed, or Oak Lab:

- the full technical paper and supplement containing NetworkIDBD equations, or
- the official NoisyMNIST/NetworkIDBD implementation plus its experiment configuration.

The source must identify the loss convention, adaptive-step-size granularity, complete weight
and meta updates, every state variable and initialization, nonlinear derivative approximation,
normalization/clipping safeguards, bias behavior, optimizer hyperparameters, horizon, evaluation
cadence, and run count. Only then is it safe to complete the symbolic derivation, verify it on a
tiny network, and begin `05_networkidbd_optimizer.ipynb`.
